# Assistiv Systems — Combined Intelligence Pipeline

Three intelligence layers for Kent & Medway in one notebook.

| Section | Cells | Output | Cadence |
|---|---|---|---|
| **FEP** Frailty Emergence Probability | 1–3 | `kent-fep-data.json` | Monthly (EPD) / Daily (Fingertips via Actions) |
| **CBI** Carer Burden Index | 4–6 | `kent-cbi-data.json` | Annually |
| **RAVI** Rural Access Vulnerability | 7–10 | `kent-ravi-data.json` | One-off + re-run when data updates |

**Before running:** Add `GITHUB_TOKEN` to Colab Secrets (🔑 left sidebar)

Each section is independent — run only the section you need.

---
*NHS Fingertips · NHSBSA EPD · ONS Census 2021 · MHCLG IMD 2019 · mySociety RUC 2021 · ONS BUA Lookup 2011*

---
## ── FEP SECTION ──────────────────────────────────────────────────────
### Frailty Emergence Probability
21 signals · 13 Kent districts · NHSBSA EPD + NHS Fingertips
Commits `kent-fep-data.json`

### Cell 1 — FEP: Install Dependencies & Configuration

In [ ]:
import subprocess
subprocess.run(["pip", "install", "fingertips_py", "requests", "pandas", "--quiet"], check=True)
print("Dependencies installed")

import requests, pandas as pd, fingertips_py as ftp
import json, csv, base64
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-fep-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f"GitHub token: {'Found' if GITHUB_TOKEN else 'MISSING - add to Colab Secrets'}")

KENT_ICB_ONS   = "E54000032"
KENT_ICB_CODE  = "QKS"
KENT_COUNTY    = "E10000016"
ENGLAND        = "E92000001"

# ── UPDATE WHEN NEW EPD MONTH PUBLISHED ─────────────────────────────────────────
# Fingertips: auto-fetches latest on every run (no update needed)
# EPD: update EPD_URL + EPD_PERIOD when NHSBSA publish a new month
# Check: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code
EPD_URL    = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
EPD_PERIOD = "Mar 2026"

# ── v4.0: UKHSA Weather-Health Alert ──────────────────────────────────────────
# The UKHSA/Met Office Weather-Health Alerting system publishes regional alert
# status as a public JSON feed. No API key required.
# Heat-Health Alerts run 1 Jun – 30 Sep; Cold-Health Alerts run 1 Nov – 30 Mar.
# South East England region code = 'E' in UKHSA taxonomy.
# Alert levels: green | yellow | amber | red
# Reference: https://www.metoffice.gov.uk/weather/warnings-and-advice/seasonal-advice/heat-health-alert-service
WEATHER_API_URL = "https://api.ukhsa.gov.uk/weather-health-alerting/alerts"
WEATHER_REGION  = "South East England"  # Matches Kent & Medway geography

# Signals with elevated clinical risk during heat (H) or cold (C) alerts.
# These flags are attached to the weather_alert block in the JSON output
# and injected into the RESILIENCE Layer 4 screening context string.
# Evidence base:
#   Heat → anticholinergic burden: impairs thermoregulation (sweating), raises toxicity risk
#   Heat → diuretics/ACE-ARB: accelerates dehydration, electrolyte disturbance
#   Cold → hypnotics/anxiolytics: CNS sedation increases fall risk on icy surfaces
#   Cold → diuretics: cold diuresis + electrolyte disruption in frail older adults
WEATHER_MED_RISK_SIGNALS = {
    "heat": ["anxiolytics", "bladder_antimusc", "antidepressants", "diuretics", "ace_arb"],
    "cold": ["hypnotics", "anxiolytics", "diuretics"],
}

# FEP score uplift applied to HIGH and CRITICAL risk districts during amber/red alerts.
# Rationale: an amber cold alert in Thanet (FEP 63) is a materially different event
# than in Tunbridge Wells (FEP 34). The uplift accelerates outreach priority.
# Uplift is time-limited — it appears in the JSON but does NOT permanently alter weights.
WEATHER_UPLIFT = {"green": 0, "yellow": 2, "amber": 5, "red": 8}  # FEP points

# England reference rates per 1,000 patients/month (OpenPrescribing 2024/25)
ENGLAND_PRESCRIBING_RATES = {
    'hypnotics':        10.2,
    'anxiolytics':       8.5,
    'antidepressants':  110.0,
    'bisphosphonates':   6.8,
    'diuretics':         2.5,
    'ace_arb':          95.0,
    'bladder_antimusc':  4.2,
    'ons_nutrition':      3.8,  # Oral nutritional supplements (BNF 090402)
                               # Source: OpenPrescribing 2024/25 — GP-level ONS items/1k pts
                               # Note: excludes tube feeds (secondary care, not in EPD)
}

# ── EPD column indices (NHSBSA EPD_SNOMED, 0-indexed) ─────────────────────────
ICB_I      = 3
BNF_I      = 14
CHEM_I     = 15
PRACTICE_I = 8
POSTCODE_I = 13
ITEMS_I    = 20
QTY_I      = 21
KENT_FILTER = 'KENT AND MEDWAY'

# ── District list sizes (NHS Digital Practice List Sizes 2024) ─────────────────
DISTRICT_LIST_SIZES = {
    "Thanet":              145_000,
    "Folkestone & Hythe":  116_000,
    "Dover":               115_000,
    "Swale":               153_000,
    "Medway":              295_000,
    "Gravesham":           106_000,
    "Ashford":             128_000,
    "Canterbury":          172_000,
    "Dartford":            111_000,
    "Maidstone":           183_000,
    "Tonbridge & Malling": 132_000,
    "Sevenoaks":           118_000,
    "Tunbridge Wells":     119_000,
}
KENT_LIST_SIZE = sum(DISTRICT_LIST_SIZES.values())
ALL_DISTRICTS  = list(DISTRICT_LIST_SIZES.keys())

# ── Postcode outward code → District lookup ────────────────────────────────────
POSTCODE_TO_DISTRICT = {
    'CT9':'Thanet','CT10':'Thanet','CT11':'Thanet','CT12':'Thanet',
    'CT1':'Canterbury','CT2':'Canterbury','CT3':'Canterbury',
    'CT4':'Canterbury','CT5':'Canterbury','CT6':'Canterbury',
    'CT13':'Dover','CT14':'Dover','CT15':'Dover','CT16':'Dover','CT17':'Dover',
    'CT18':'Folkestone & Hythe','CT19':'Folkestone & Hythe',
    'CT20':'Folkestone & Hythe','CT21':'Folkestone & Hythe','TN29':'Folkestone & Hythe',
    'ME1':'Medway','ME2':'Medway','ME3':'Medway','ME4':'Medway',
    'ME5':'Medway','ME7':'Medway','ME8':'Medway',
    'ME9':'Swale','ME10':'Swale','ME11':'Swale','ME12':'Swale','ME13':'Swale',
    'ME14':'Maidstone','ME15':'Maidstone','ME16':'Maidstone',
    'ME17':'Maidstone','ME18':'Maidstone',
    'ME19':'Tonbridge & Malling','ME20':'Tonbridge & Malling',
    'TN10':'Tonbridge & Malling','TN11':'Tonbridge & Malling','TN12':'Tonbridge & Malling',
    'DA11':'Gravesham','DA12':'Gravesham','DA13':'Gravesham',
    'DA1':'Dartford','DA2':'Dartford',
    'TN13':'Sevenoaks','TN14':'Sevenoaks','TN15':'Sevenoaks',
    'TN16':'Sevenoaks','TN8':'Sevenoaks','DA3':'Sevenoaks','DA4':'Sevenoaks',
    'TN1':'Tunbridge Wells','TN2':'Tunbridge Wells','TN3':'Tunbridge Wells',
    'TN4':'Tunbridge Wells','TN5':'Tunbridge Wells','TN6':'Tunbridge Wells',
    'TN23':'Ashford','TN24':'Ashford','TN25':'Ashford',
    'TN26':'Ashford','TN27':'Ashford','TN30':'Ashford',
}

print(f"Config v5.0 loaded | ICB: {KENT_ICB_ONS} | EPD: {EPD_PERIOD}")
print(f"District list sizes: {len(DISTRICT_LIST_SIZES)} districts, {KENT_LIST_SIZE:,} total patients")
print(f"Postcode lookup: {len(POSTCODE_TO_DISTRICT)} outward codes → {len(set(POSTCODE_TO_DISTRICT.values()))} districts")
print(f"Weather API: {WEATHER_API_URL}")

### Cell 2 — FEP: Fetch NHS Fingertips Indicators

In [ ]:
# ── v4.0: Fetch UKHSA Weather-Health Alert ────────────────────────────────────
# The UKHSA WHA JSON API is a public endpoint — no key required.
# Falls back gracefully if unreachable.

def fetch_weather_alert():
    """Fetch current UKHSA Weather-Health Alert for South East England.
    Returns a dict with alert_level, alert_type, region, fetched_at, and derived fields.
    Fails safe to green/none if API is unreachable."""

    now_utc   = datetime.now(timezone.utc)
    month     = now_utc.month
    season    = "heat" if 6 <= month <= 9 else "cold" if month in (11,12,1,2,3) else "none"

    defaults = {
        "alert_level":      "green",
        "alert_type":       season,
        "region":           WEATHER_REGION,
        "fetched_at":       now_utc.isoformat(),
        "source":           "UKHSA Weather-Health Alerting / Met Office",
        "api_status":       "fallback",
        "med_risk_signals": [],
        "fep_uplift":       0,
        "layer4_context":   "",
    }

    try:
        resp = requests.get(
            WEATHER_API_URL,
            params={"region": WEATHER_REGION},
            timeout=10,
            headers={"Accept": "application/json", "User-Agent": "AssistivSystems/4.0"}
        )

        if resp.status_code == 200:
            data   = resp.json()
            alerts = data if isinstance(data, list) else data.get("alerts", [])

            # Find the most severe active alert for South East England
            level_order = {"red": 4, "amber": 3, "yellow": 2, "green": 1}
            best_level  = "green"
            best_type   = season

            for alert in alerts:
                region = (alert.get("region") or alert.get("area") or "").lower()
                if "south east" not in region and "kent" not in region:
                    continue
                lvl  = (alert.get("level") or alert.get("alert_level") or "green").lower()
                atype = (alert.get("type") or alert.get("alert_type") or season).lower()
                if level_order.get(lvl, 0) > level_order.get(best_level, 0):
                    best_level = lvl
                    best_type  = atype

            defaults["api_status"] = "live"
            defaults["alert_level"] = best_level
            defaults["alert_type"]  = best_type

        else:
            # API reachable but unexpected status — try scraping the UKHSA alert page
            print(f"  WHA API returned {resp.status_code} — attempting HTML fallback")
            page = requests.get(
                "https://www.ukhsa.gov.uk/weather-health-alerting",
                timeout=10, headers={"User-Agent": "AssistivSystems/4.0"}
            )
            text = page.text.lower()
            for lvl in ("red", "amber", "yellow"):
                if f"south east" in text and lvl in text:
                    defaults["alert_level"] = lvl
                    defaults["api_status"]  = "html_fallback"
                    break

    except Exception as e:
        print(f"  WHA API unreachable: {e} — defaulting to green/no alert")
        defaults["api_status"] = f"error: {str(e)[:80]}"

    # ── Derive fields from confirmed alert_level + alert_type ──────────────────
    level = defaults["alert_level"]
    atype = defaults["alert_type"]
    uplift = WEATHER_UPLIFT.get(level, 0)

    # Which medication signals have elevated clinical risk right now?
    med_risk = WEATHER_MED_RISK_SIGNALS.get(atype, []) if level in ("yellow","amber","red") else []

    # Plain-English context string for RESILIENCE Layer 4 injection
    if level == "green" or atype == "none":
        ctx = ""
    else:
        level_desc = {"yellow":"Yellow", "amber":"Amber", "red":"Red"}.get(level, level.title())
        type_desc  = "Heat-Health" if atype == "heat" else "Cold-Health"
        sigs_str   = ", ".join(med_risk) if med_risk else "none flagged"
        if atype == "heat":
            med_note = ("Anticholinergic burden (antidepressants, anxiolytics, bladder antimuscarinics) "
                        "is elevated during heat — impaired thermoregulation and dehydration risk. "
                        "Diuretic and ACE/ARB users face accelerated electrolyte depletion. "
                        "Apply heightened sensitivity to medication-related responses.")
        else:
            med_note = ("Hypnotics and anxiolytics increase fall risk on icy or wet surfaces. "
                        "Cold diuresis compounds electrolyte risk in diuretic users. "
                        "Apply heightened sensitivity to mobility, falls, and medication responses.")
        ctx = (
            f"WEATHER ALERT ACTIVE: UKHSA/Met Office {level_desc} {type_desc} Alert — "
            f"South East England (current as of {now_utc.strftime('%Y-%m-%d %H:%M')} UTC). "
            f"Elevated medication risk signals: {sigs_str}. {med_note}"
        )

    defaults["med_risk_signals"] = med_risk
    defaults["fep_uplift"]       = uplift
    defaults["layer4_context"]   = ctx

    return defaults


print("Fetching UKHSA Weather-Health Alert...")
weather_alert = fetch_weather_alert()

print(f"  Status:        {weather_alert['api_status']}")
print(f"  Alert level:   {weather_alert['alert_level'].upper()}")
print(f"  Alert type:    {weather_alert['alert_type']}")
print(f"  FEP uplift:    +{weather_alert['fep_uplift']} points (applied to high/critical districts)")
print(f"  Med risk:      {weather_alert['med_risk_signals'] or 'none'}")
if weather_alert['layer4_context']:
    print(f"\n  Layer 4 context string:")
    print(f"  {weather_alert['layer4_context']}")
else:
    print(f"  Layer 4 context: none (green alert)")

### Cell 3 — FEP: Stream NHSBSA EPD · Calculate Scores · Commit

In [ ]:
FINGERTIPS_INDICATORS = {
    'falls_65':           (22401, 'Falls admissions 65+',        KENT_ICB_ONS),
    'falls_65_79':        (22402, 'Falls admissions 65-79',      KENT_ICB_ONS),
    'falls_80':           (22403, 'Falls admissions 80+',        KENT_ICB_ONS),
    'winter_mortality':   (90360, 'Winter mortality index',      KENT_COUNTY),
    'loneliness':         (94175, 'Loneliness often/always',     KENT_ICB_ONS),
    'social_isolation':   (90280, 'Social isolation - SC users', KENT_COUNTY),
    'dementia_diagnosis': (92949, 'Dementia diagnosis rate 65+', KENT_ICB_ONS),
    'hip_fractures_65':   (41401, 'Hip fractures 65+',           KENT_ICB_ONS),
    'hip_fractures_80':   (41403, 'Hip fractures 80+',           KENT_ICB_ONS),
    'fuel_poverty':       (93759, 'Fuel poverty',                KENT_ICB_ONS),
}

fingertips_results = {}
print("Fetching NHS Fingertips indicators...")
print(f"{'Indicator':<35} {'Kent':>8}  {'England':>8}  Period")
print("-" * 70)

for key, (ind_id, label, area_code) in FINGERTIPS_INDICATORS.items():
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None:
            raise ValueError("Returned None")
        kent = data[data['Area Code'] == area_code].sort_values('Time period')
        eng  = data[data['Area Code'] == ENGLAND].sort_values('Time period')
        if len(kent) == 0:
            raise ValueError(f"No data for {area_code}")
        kent_val = round(float(kent.tail(1)['Value'].values[0]), 2)
        eng_val  = round(float(eng.tail(1)['Value'].values[0]), 2) if len(eng) else None
        period   = str(kent.tail(1)['Time period'].values[0])
        fingertips_results[key] = {
            'value': kent_val, 'england': eng_val,
            'period': period, 'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }
        direction = "▲" if eng_val and kent_val > eng_val else "▼"
        print(f"  {label:<33} {kent_val:>8}  {str(eng_val):>8}  {period}  {direction}")
    except Exception as e:
        print(f"  FAILED {label:<29} -- {e}")
        fingertips_results[key] = {
            'value': None, 'england': None, 'period': None,
            'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }

real_ft = sum(1 for v in fingertips_results.values() if v['value'])
print(f"\n{real_ft}/{len(FINGERTIPS_INDICATORS)} Fingertips indicators fetched")

### Cell 3b — FEP: NHS 111 Monthly Data Fetch
Run monthly when new IUCADC data is published.

In [ ]:
TARGET_BNF = (
    '0401010',   # Hypnotics
    '0401020',   # Anxiolytics
    '0403',      # Antidepressants (incl anticholinergic tricyclics)
    '060602',    # Bisphosphonates
    '0201',      # Diuretics
    '0205',      # ACE inhibitors / ARBs
    '0704010',   # Bladder antimuscarinics
    '090402',    # Oral nutritional supplements (Fortisip, Ensure, Fresubin etc.) ← NEW v5
)

def bnf_to_signal(bnf):
    if   bnf.startswith('0401010'): return 'hypnotics'
    elif bnf.startswith('0401020'): return 'anxiolytics'
    elif bnf.startswith('0403'):    return 'antidepressants'
    elif bnf.startswith('060602'):  return 'bisphosphonates'
    elif bnf.startswith('0201'):    return 'diuretics'
    elif bnf.startswith('0205'):    return 'ace_arb'
    elif bnf.startswith('0704010'): return 'bladder_antimusc'
    elif bnf.startswith('090402'):  return 'ons_nutrition'   # ← NEW v5
    return None

district_signal_items = defaultdict(lambda: defaultdict(int))
district_signal_qty   = defaultdict(lambda: defaultdict(float))
signal_items    = defaultdict(int)
signal_quantity = defaultdict(float)
signal_drugs    = defaultdict(set)
practice_district_cache = {}
unmapped_practices = set()
unmapped_items = 0
rows_read = kent_rows = mapped_rows = 0
buffer = ""; header_done = False

print(f"Streaming NHSBSA EPD — {EPD_PERIOD} (v5.0 — 8 BNF groups incl ONS nutrition)")
print(f"Targeting {len(TARGET_BNF)} BNF groups. Takes 3–5 minutes.\n")

try:
    with requests.get(EPD_URL, stream=True, timeout=300,
                      headers={"User-Agent": "Mozilla/5.0 AssistivSystems/4.0"}) as resp:
        print(f"HTTP {resp.status_code}")
        if resp.status_code != 200:
            raise ValueError(f"HTTP {resp.status_code} — check EPD_URL in Cell 1")
        for raw_chunk in resp.iter_content(chunk_size=512*1024):
            buffer += raw_chunk.decode('utf-8', errors='replace')
            lines = buffer.split('\n'); buffer = lines[-1]
            for line in lines[:-1]:
                line = line.strip()
                if not line: continue
                if not header_done:
                    header_done = True; continue
                rows_read += 1
                if rows_read % 2_000_000 == 0:
                    print(f"  {rows_read:>12,} rows | {kent_rows:>6,} Kent | "
                          f"{mapped_rows:>6,} mapped | {len(practice_district_cache)} practices seen")
                if KENT_FILTER not in line: continue
                try:
                    row = next(csv.reader([line]))
                except Exception:
                    continue
                if len(row) <= max(ICB_I, BNF_I, ITEMS_I, QTY_I, PRACTICE_I, POSTCODE_I): continue
                if KENT_FILTER not in row[ICB_I]: continue
                bnf = row[BNF_I].strip('"')
                if not bnf.startswith(TARGET_BNF): continue
                signal = bnf_to_signal(bnf)
                if not signal: continue
                try:
                    items         = int(row[ITEMS_I])
                    qty           = float(row[QTY_I])
                    chem          = row[CHEM_I]
                    practice_code = row[PRACTICE_I].strip()
                    postcode      = row[POSTCODE_I].strip().upper()
                except (ValueError, IndexError):
                    continue
                kent_rows += 1
                signal_items[signal]    += items
                signal_quantity[signal] += qty
                signal_drugs[signal].add(chem)
                if practice_code not in practice_district_cache:
                    outward  = postcode.split()[0] if postcode else ''
                    district = POSTCODE_TO_DISTRICT.get(outward)
                    practice_district_cache[practice_code] = district
                    if district is None:
                        unmapped_practices.add(f"{practice_code}({outward})")
                district = practice_district_cache[practice_code]
                if district:
                    district_signal_items[district][signal] += items
                    district_signal_qty[district][signal]   += qty
                    mapped_rows += 1
                else:
                    unmapped_items += items
except Exception as e:
    import traceback; print(f"\nError at row {rows_read:,}: {e}"); traceback.print_exc()

print(f"\nCOMPLETE — {rows_read:,} rows scanned, {kent_rows:,} Kent matched")
print(f"  District-mapped: {mapped_rows:,} ({100*mapped_rows/max(kent_rows,1):.1f}%)")
print(f"  Unmapped items:  {unmapped_items:,}")
print(f"  Practices:       {len(practice_district_cache)} seen | {len(unmapped_practices)} unmapped")
if unmapped_practices:
    print(f"  Unmapped list: {sorted(unmapped_practices)[:20]}")

In [ ]:
EPD_SIGNAL_KEYS = [
    'hypnotics', 'anxiolytics', 'antidepressants',
    'bisphosphonates', 'diuretics', 'ace_arb', 'bladder_antimusc',
    'ons_nutrition',  # ← NEW v5 — oral nutritional supplements (BNF 090402)
]

epd_results = {}
print("ICB-level rates (for comparison):")
print(f"  {'Signal':<22} {'Kent/1k':>9}  {'Eng/1k':>9}  {'Ratio':>7}")
print(f"  {'-'*55}")
for key in EPD_SIGNAL_KEYS:
    items  = signal_items.get(key, 0)
    qty    = signal_quantity.get(key, 0.0)
    drugs  = len(signal_drugs.get(key, set()))
    kent_r = round((items / KENT_LIST_SIZE) * 1000, 3) if items else 0
    eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
    ratio  = round(kent_r / eng_r, 3) if eng_r and kent_r else 0
    estimate_flag = " [est]" if key in ('anxiolytics', 'bladder_antimusc') else ""
    epd_results[key] = {
        'rate_per_1000': kent_r, 'england': eng_r, 'items': items,
        'qty': round(qty), 'substances': drugs, 'ratio': ratio,
        'period': EPD_PERIOD,
        'source': f'NHSBSA EPD EPD_SNOMED_{EPD_PERIOD.replace(" ","").upper()}',
        'england_note': 'literature estimate' if estimate_flag else 'OpenPrescribing 2024/25',
    }
    print(f"  {key:<22} {kent_r:>9.3f}  {eng_r:>9.1f}{estimate_flag}  {ratio:>7.3f}")

print(f"\nDistrict prescribing rates per 1,000 patients:")
header = f"  {'District':<25} {'List':>7}"
for s in EPD_SIGNAL_KEYS:
    header += f"  {s[:7]:>8}"
print(header); print(f"  {'-'*110}")

epd_district_results = {}
for district in ALL_DISTRICTS:
    list_size = DISTRICT_LIST_SIZES[district]
    epd_district_results[district] = {}
    row_str = f"  {district:<25} {list_size:>7,}"
    for key in EPD_SIGNAL_KEYS:
        items  = district_signal_items[district].get(key, 0)
        rate   = round((items / list_size) * 1000, 3) if items else 0
        eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
        ratio  = round(rate / eng_r, 3) if eng_r and rate else 0
        epd_district_results[district][key] = {
            'rate_per_1000': rate, 'england': eng_r,
            'items': items, 'ratio': ratio,
            'period': EPD_PERIOD, 'source': f'NHSBSA EPD practice-level — {EPD_PERIOD}',
        }
        flag = "▲" if ratio > 1.05 else "▼" if ratio < 0.95 else "="
        row_str += f"  {rate:>7.2f}{flag}"
    print(row_str)

print("\nQA — district signals with ratio > 2.0 or < 0.3 vs England:")
issues = []
for district in ALL_DISTRICTS:
    for key in EPD_SIGNAL_KEYS:
        r = epd_district_results[district][key].get('ratio', 0)
        if r > 2.0 or (0 < r < 0.3):
            issues.append(f"  *** {district} / {key}: ratio {r:.3f} — review list size or BNF filter")
if issues:
    for i in issues: print(i)
else:
    print("  All district rates within plausible range ✓")

# ONS-specific QA — wider ratio threshold reflects genuine practice variation
ons_ratio = epd_results.get("ons_nutrition", {}).get("ratio", 0)
if ons_ratio > 0:
    if ons_ratio > 4.0 or ons_ratio < 0.2:
        print(f"  *** ONS ratio {ons_ratio:.3f} — unusually wide, check BNF 090402 filter")
    else:
        print(f"  ONS nutrition ratio vs England: {ons_ratio:.3f} ✓ (expected range 0.2–4.0)")

real_epd = sum(1 for v in epd_results.values() if v['rate_per_1000'] > 0)
print(f"\n{real_epd}/{len(EPD_SIGNAL_KEYS)} EPD signals computed")

In [ ]:
# ── SIGNAL DEFINITIONS ────────────────────────────────────────────────────────
SIGNAL_NAMES = [
    "Over-75s Living Alone",       # 0   synthetic (modelled from ONS Census)
    "Falls Admissions 65+",        # 1   Fingertips real
    "Hip Fracture Rate 65+",       # 2   Fingertips real
    "Deprivation (IMD)",           # 3   synthetic (modelled from IMD 2019)
    "Winter Mortality Index",      # 4   Fingertips real
    "Care Home Gap",               # 5   synthetic (modelled from CQC register)
    "Loneliness Rate",             # 6   Fingertips real
    "Dementia Diagnosis Rate",     # 7   Fingertips real (inverted — lower = worse)
    "Hip Fractures 80+",           # 8   Fingertips real
    "Social Isolation Rate",       # 9   Fingertips real
    "Hypnotics Prescribing",       # 10  EPD district-level
    "Antidepressant Rate",         # 11  EPD district-level
    "Bisphosphonate Rate",         # 12  EPD district-level
    "Diuretics Rate",              # 13  EPD district-level
    "ACE/ARB Prescribing",         # 14  EPD district-level
    "Anxiolytics Prescribing",     # 15  EPD district-level
    "Bladder Antimuscarinic Rate", # 16  EPD district-level
    "Oral Nutritional Supplements",# 17  EPD district-level ← NEW v5 (BNF 090402)
]

EPD_SIGNAL_INDICES      = list(range(10, 18))  # now 8 EPD signals
EPD_SIGNAL_KEYS_ORDERED = [
    'hypnotics', 'antidepressants', 'bisphosphonates',
    'diuretics', 'ace_arb', 'anxiolytics', 'bladder_antimusc',
    'ons_nutrition',  # ← NEW v5
]

WEIGHTS = [
    0.13,  # alone_75
    0.12,  # falls_65
    0.09,  # hip_65
    0.08,  # deprivation
    0.08,  # winter_mortality
    0.07,  # care_home_gap
    0.06,  # loneliness
    0.06,  # dementia
    0.05,  # hip_80
    0.05,  # social_isolation
    0.05,  # hypnotics
    0.05,  # antidepressants
    0.04,  # bisphosphonates
    0.00,  # diuretics — weight removed in v5; signal retained for reporting
    0.00,  # ace_arb — weight removed in v5 alongside diuretics (retained for reporting)
    0.03,  # anxiolytics
    0.01,  # bladder_antimusc (reduced from 2% in v5)
    0.03,  # ons_nutrition ← NEW v5 — direct FRAIL Scale (Loss of weight) signal
]
assert abs(sum(WEIGHTS) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS):.4f} — must be 1.0"
print(f"Weight check: {sum(WEIGHTS):.4f} ✓")

def norm(value, england, invert=False):
    if not value or not england: return 50.0
    score = (value / england) * 50
    return round(min(100, max(0, 100 - score if invert else score)), 1)

def ft(key, invert=False):
    v = fingertips_results.get(key, {})
    return norm(v.get('value'), v.get('england'), invert)

def epd_for_district(district, signal_key):
    v = epd_district_results.get(district, {}).get(signal_key, {})
    return norm(v.get('rate_per_1000'), ENGLAND_PRESCRIBING_RATES.get(signal_key))

ICB_BASE_NONEEPD = [
    50.0,
    ft('falls_65'),
    ft('hip_fractures_65'),
    50.0,
    ft('winter_mortality'),
    50.0,
    ft('loneliness'),
    ft('dementia_diagnosis', invert=True),
    ft('hip_fractures_80'),
    ft('social_isolation'),
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,  # EPD signals — set in district loop
]

print("\nICB normalised scores for non-EPD signals (England avg = 50):")
for i, (name, score) in enumerate(zip(SIGNAL_NAMES[:10], ICB_BASE_NONEEPD[:10])):
    is_real = score != 50.0
    tag = "REAL  " if is_real else "MODELLED"
    bar = ">" if score > 50 else "<" if score < 50 else "="
    print(f"  {bar} {score:>5.1f}  [{tag}]  {name}")

# v5.0: PROFILES extended to 18 signals.
# The new ONS signal (index 17) uses 1 as the profile multiplier for all districts —
# EPD district-level data replaces the profile multiplier at scoring time, same as
# the other EPD signals (indices 10–16). The profile value is a passthrough placeholder.
PROFILES = {
    "Thanet":              [1.30,1.25,1.20,1.35,1.28,1.30,1.25,1.18,1.20,1.22, 1,1,1,1,1,1,1,1],
    "Folkestone & Hythe":  [1.22,1.18,1.15,1.22,1.20,1.20,1.18,1.12,1.15,1.15, 1,1,1,1,1,1,1,1],
    "Dover":               [1.18,1.15,1.12,1.18,1.15,1.10,1.14,1.08,1.10,1.10, 1,1,1,1,1,1,1,1],
    "Swale":               [1.12,1.10,1.08,1.12,1.10,1.12,1.08,1.05,1.08,1.05, 1,1,1,1,1,1,1,1],
    "Medway":              [1.06,1.05,1.04,1.08,1.05,1.08,1.02,1.02,1.05,1.02, 1,1,1,1,1,1,1,1],
    "Gravesham":           [1.00,0.98,1.02,1.02,1.00,1.05,0.98,1.00,1.02,1.00, 1,1,1,1,1,1,1,1],
    "Ashford":             [0.96,0.95,0.98,0.98,0.96,1.00,0.94,0.96,0.98,0.95, 1,1,1,1,1,1,1,1],
    "Canterbury":          [0.90,0.90,0.92,0.85,0.92,0.92,0.92,0.90,0.90,0.90, 1,1,1,1,1,1,1,1],
    "Dartford":            [0.88,0.88,0.90,0.95,0.88,0.90,0.88,0.88,0.88,0.88, 1,1,1,1,1,1,1,1],
    "Maidstone":           [0.85,0.85,0.88,0.95,0.85,0.92,0.85,0.85,0.85,0.85, 1,1,1,1,1,1,1,1],
    "Tonbridge & Malling": [0.78,0.78,0.82,0.78,0.80,0.85,0.80,0.80,0.80,0.80, 1,1,1,1,1,1,1,1],
    "Sevenoaks":           [0.65,0.65,0.68,0.52,0.65,0.75,0.65,0.65,0.65,0.65, 1,1,1,1,1,1,1,1],
    "Tunbridge Wells":     [0.58,0.58,0.62,0.58,0.60,0.65,0.60,0.60,0.60,0.60, 1,1,1,1,1,1,1,1],
}

LAD_CODES = {
    "Thanet":"E07000114", "Folkestone & Hythe":"E07000112", "Dover":"E07000108",
    "Swale":"E07000113",  "Medway":"E06000035",  "Gravesham":"E07000109",
    "Ashford":"E07000105","Canterbury":"E07000106","Dartford":"E07000107",
    "Maidstone":"E07000110","Tonbridge & Malling":"E07000115",
    "Sevenoaks":"E07000111","Tunbridge Wells":"E07000116",
}

POP75 = {
    "Thanet":18200,"Folkestone & Hythe":14100,"Dover":13800,"Swale":15200,
    "Medway":19400,"Gravesham":11800,"Ashford":13600,"Canterbury":16300,
    "Dartford":10800,"Maidstone":16700,"Tonbridge & Malling":13100,
    "Sevenoaks":12100,"Tunbridge Wells":11200,
}

# ── v4.0: Weather uplift config ────────────────────────────────────────────────
# Applied to districts scoring 'high' or 'critical' during amber/red alerts.
# fep_weather is stored alongside fep (base) — the map can display either.
# The uplift is not baked into weights; it is additive and labelled separately.
w_level  = weather_alert["alert_level"]
w_type   = weather_alert["alert_type"]
w_uplift = weather_alert["fep_uplift"]
w_active = w_level in ("yellow", "amber", "red")

if w_active:
    print(f"\n⚠ Weather alert active: {w_level.upper()} {w_type} — "
          f"+{w_uplift} FEP uplift on high/critical districts")
    print(f"  Elevated medication risk: {weather_alert['med_risk_signals']}")
else:
    print(f"\nWeather: {w_level.upper()} — no uplift applied")

# ── BUILD DISTRICT SCORES ──────────────────────────────────────────────────────
districts = []
print(f"\nDistrict FEP scores (v4.0):")
print(f"  {'District':<25} {'FEP':>5}  {'W-FEP':>6}  {'Risk':<10}  antidep  hypnot  anxiol")
print("  " + "-" * 80)

for name, profile in PROFILES.items():
    signals = [
        round(min(100, max(0, ICB_BASE_NONEEPD[i] * profile[i])), 1)
        for i in range(18)
    ]
    for idx, epd_key in zip(EPD_SIGNAL_INDICES, EPD_SIGNAL_KEYS_ORDERED):
        signals[idx] = epd_for_district(name, epd_key)

    fep_base = round(min(100, max(0, sum(s * w for s, w in zip(signals, WEIGHTS)))))
    risk     = "critical" if fep_base >= 70 else "high" if fep_base >= 55 else "moderate" if fep_base >= 40 else "low"

    # v4.0: weather uplift applied only to high/critical districts during active alerts
    fep_weather = fep_base
    if w_active and risk in ("high", "critical"):
        fep_weather = min(100, fep_base + w_uplift)

    # Recalculate displayed risk using weather-uplifted score for map display
    risk_weather = "critical" if fep_weather >= 70 else "high" if fep_weather >= 55 else "moderate" if fep_weather >= 40 else "low"

    ad = epd_district_results[name]['antidepressants']['rate_per_1000']
    hy = epd_district_results[name]['hypnotics']['rate_per_1000']
    ax = epd_district_results[name]['anxiolytics']['rate_per_1000']
    uplift_marker = f" +{w_uplift}⚠" if fep_weather > fep_base else ""
    print(f"  {name:<25} {fep_base:>5}  {fep_weather:>6}{uplift_marker:<4}  {risk:<10}  {ad:>7.2f}  {hy:>6.2f}  {ax:>6.2f}")

    districts.append({
        "name":          name,
        "lad_code":      LAD_CODES[name],
        "fep":           fep_base,         # base score — stable, weather-independent
        "fep_weather":   fep_weather,      # weather-uplifted score — use for live map display
        "risk":          risk,             # based on fep_base
        "risk_weather":  risk_weather,     # based on fep_weather
        "signals":       signals,
        "signal_names":  SIGNAL_NAMES,
        "pop75":         POP75[name],
        "list_size":     DISTRICT_LIST_SIZES[name],
        "epd_district":  {k: epd_district_results[name][k] for k in EPD_SIGNAL_KEYS},
    })

districts.sort(key=lambda x: x["fep_weather"], reverse=True)

# ── Scorecard vs previous commit ───────────────────────────────────────────────
print("\n── v4.0 vs previous committed scores ──")
try:
    prev = requests.get(
        f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}",
        timeout=10
    ).json()
    prev_scores = {d['name']: d['fep'] for d in prev.get('districts', [])}
    print(f"  {'District':<25} {'prev':>5}  {'v4.0':>5}  {'Δ':>5}  {'W-FEP':>6}")
    print(f"  {'-'*55}")
    for d in districts:
        n  = d['name']
        v2 = prev_scores.get(n, '?')
        v4 = d['fep']
        vw = d['fep_weather']
        delta = (v4 - v2) if isinstance(v2, int) else '?'
        print(f"  {n:<25} {str(v2):>5}  {v4:>5}  {str(delta):>5}  {vw:>6}")
except Exception as e:
    print(f"  (Could not fetch previous scores: {e})")

# ── ASSEMBLE OUTPUT ────────────────────────────────────────────────────────────
real_signals = (
    [k for k, v in fingertips_results.items() if v.get('value')] +
    [k for k, v in epd_results.items() if v.get('rate_per_1000', 0) > 0]
)

output = {
    "meta": {
        "generated":          datetime.now(timezone.utc).isoformat(),
        "description":        "Kent & Medway FEP scores — Assistiv Systems Layer 2",
        "version":            "5.0 — oral nutritional supplements signal added",
        "icb":                "NHS Kent and Medway ICB (QKS)",
        "icb_ons_code":       KENT_ICB_ONS,
        # CORRECTED: 14 real signals, 3 modelled — not '17 real'
        "signals_real_count": len(real_signals),
        "signals_modelled_count": 3,
        "data_quality":       f"{len(real_signals)} real signals · 3 modelled (synthetic) of 18 total",
        "signals_real":       real_signals,
        "signals_modelled":   ["alone_75", "deprivation_imd", "care_home_gap"],
        "signal_names":       SIGNAL_NAMES,
        "weights":            WEIGHTS,
        "sources": {
            "fingertips":     "NHS Fingertips/OHID PHOF via fingertips_py",
            "epd":            f"NHSBSA EPD practice-level — {EPD_PERIOD}",
            "epd_method":     "Practice POSTCODE (col 13) → district via POSTCODE_TO_DISTRICT",
            "demographic":    "ONS Census 2021 · IMD 2019 · CQC register",
            "weather":        "UKHSA Weather-Health Alerting / Met Office",
        },
        "new_in_v4": [
            "UKHSA/Met Office Weather-Health Alert fetched at run time",
            "fep_weather field: weather-uplifted FEP for live map display",
            "weather_alert block added to JSON root for Layer 4 context injection",
            "data_quality metadata corrected: 14 real, 3 modelled (was incorrectly '17 real')",
        ],
        "new_in_v5": [
            "Signal 18: ons_nutrition — BNF 090402 oral nutritional supplements",
            "Weight 3% — direct FRAIL Scale (Loss of weight) clinical signal",
            "Funded by: bladder_antimusc 2%→1%, diuretics 1%→0% (retained for reporting)",
            "England reference rate: 3.8 per 1,000 patients/month (OpenPrescribing 2024/25)",
        ],
        "note": (
            "fep = base score (weather-independent, use for longitudinal comparison). "
            "fep_weather = live score including weather uplift (use for operational map display). "
            "EPD: district-level actual. Fingertips + modelled: ICB × demographic profile. "
            "Phase 2: MSOA Fingertips linkage."
        ),
    },
    # v4.0: weather_alert block at JSON root — consumed by Layer 4 RESILIENCE
    "weather_alert": weather_alert,
    "icb_baseline": {
        "fingertips":  fingertips_results,
        "prescribing": epd_results,
    },
    "districts": districts,
}

# ── COMMIT TO GITHUB ───────────────────────────────────────────────────────────
def commit_to_github(content, repo, filepath, token):
    api_url = f"https://api.github.com/repos/{repo}/contents/{filepath}"
    headers = {"Authorization": "token " + token,
               "Accept": "application/vnd.github.v3+json"}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None
    w_tag = f" [{w_level.upper()} {w_type}]" if w_active else ""
    payload = {
        "message": (
            f"FEP v5.0 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — "
            f"18 signals incl ONS nutrition ({len(practice_district_cache)} practices){w_tag}"
        ),
        "content": b64,
        "branch":  "main",
    }
    if sha: payload["sha"] = sha
    r = requests.put(api_url, headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"\nCommitted to GitHub ✓")
        print(f"  File:   https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}")
        print(f"  Commit: {r.json().get('commit',{}).get('html_url','')}")
        return True
    print(f"\nCommit failed: {r.status_code} — {r.json().get('message','')}")
    return False

print("\nCommitting to GitHub...")
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)

# ── HISTORIC SNAPSHOT ────────────────────────────────────────────────────────
# Also commit a dated copy to history/ for trend analysis
today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
HISTORY_FILE = f"history/kent-fep-{today}.json"

print(f"\nCommitting historic snapshot: {HISTORY_FILE}...")
result = commit_to_github(output, GITHUB_REPO, HISTORY_FILE, GITHUB_TOKEN)
if result:
    print(f"  Historic snapshot saved: {HISTORY_FILE}")
else:
    print("  Historic snapshot failed — current JSON still updated")

print("\nDone. v5.0 — 18 signals including ONS nutrition (BNF 090402).")
print("Run whenever a new EPD month is published (monthly). Fingertips auto-fetches latest on every run.")
print(f"Practices mapped: {len(practice_district_cache)} | Unmapped: {len(unmapped_practices)}")
print(f"Weather: {w_level.upper()} {w_type} | FEP uplift applied: {'+'+str(w_uplift) if w_active else 'none'}")

In [ ]:
!pip install beautifulsoup4 --quiet

"""
fetch_111_data.py — Assistiv Systems NHS 111 Monthly Data Fetcher
=================================================================
Scrapes NHS England IUCADC page, downloads the latest Provisional CSV,
extracts Kent/Medway/Sussex contract figures (SECAmb 111AI9),
applies population weighting to estimate Kent & Medway ICB share,
and commits:
  - kent-111-data.json              (current — always overwritten)
  - history/kent-111-YYYY-MM.json   (monthly snapshot — never overwritten)

Data format: long/narrow — one row per day per item code.
Key item codes: A01=calls received, A03=answered, E02=amb dispatch,
                E03=ED referral, G03=GP referral, E18=treated/advised.

Geography: SECAmb contract "Kent, Medway & Sussex" — Kent ICB is
approximately 55% of this population. A weighting factor is applied.

No manual updates needed — scrapes page to find randomised CSV URLs.
"""

import os, re, json, base64, requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
from datetime import datetime, timezone

# ── CONSTANTS ─────────────────────────────────────────────────────────
GITHUB_REPO  = "silegrand/assistivagents"

# SECAmb contract area: Kent, Medway & Sussex
# Kent & Medway ICB population as % of total contract area population
# Kent ICB: 1.9m / (Kent 1.9m + Surrey 1.2m + Sussex 1.7m) ≈ 0.40
# Kent, Medway & Sussex contract = Kent + East Sussex + West Sussex + Brighton
# Kent ICB 1.9m / contract area ~4.85m = 0.39 — use 0.40 as conservative estimate
KENT_WEIGHT  = 0.40            # Kent ICB share of SECAmb contract
KENT_ICB_POP = 1_900_000
ENGLAND_POP  = 56_490_000

# England reference rates (published IUCADC national 2024/25 annualised)
# Per 1,000 population per month
ENG_CALL_RATE_MONTHLY = 65.4   # ~785/year
ENG_AMB_RATE_MONTHLY  =  3.9   # ~47/year
ENG_ED_RATE_MONTHLY   =  7.7   # ~92/year

IUCADC_PAGES = [
    "https://www.england.nhs.uk/statistics/statistical-work-areas/iucadc-new-from-april-2021/"
    "integrated-urgent-care-aggregate-data-collection-iucadc-inc-nhs111-statistics-apr-2026-mar-2027/",
    "https://www.england.nhs.uk/statistics/statistical-work-areas/iucadc-new-from-april-2021/"
    "integrated-urgent-care-aggregate-data-collection-iucadc-including-nhs111-statistics-apr-2025-mar-2026/",
]

MONTH_MAP = {
    'jan':'01','feb':'02','mar':'03','apr':'04','may':'05','jun':'06',
    'jul':'07','aug':'08','sep':'09','oct':'10','nov':'11','dec':'12',
}
YEAR_MAP = {'25':'2025','26':'2026','27':'2027','28':'2028'}
HEADERS  = {"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}


# ── STEP 1: SCRAPE PAGE ───────────────────────────────────────────────
def discover_latest_csv():
    print("Step 1: Scraping NHS England IUCADC pages...")
    url_pattern = re.compile(
        r'Provisional-IUCADC-Raw-([A-Za-z]{3})(\d{2})',
        re.IGNORECASE
    )
    candidates = []

    for page_url in IUCADC_PAGES:
        try:
            r = requests.get(page_url, timeout=20, headers=HEADERS)
            if r.status_code != 200:
                continue
            soup = BeautifulSoup(r.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a['href']
                if 'Provisional-IUCADC-Raw' not in href:
                    continue
                if not href.endswith('.csv'):
                    continue
                m = url_pattern.search(href)
                if m:
                    mon, yr = m.groups()
                    month_num = MONTH_MAP.get(mon.lower())
                    year = YEAR_MAP.get(yr, f"20{yr}")
                    if month_num:
                        period = f"{year}-{month_num}"
                        period_nice = f"{mon.capitalize()} {year}"
                        full_url = href if href.startswith('http') else \
                                   'https://www.england.nhs.uk' + href
                        candidates.append((period, period_nice, full_url))
                        print(f"  Found: {period_nice} — ...{href[-45:]}")
        except Exception as e:
            print(f"  Warning: {e}")

    if not candidates:
        raise RuntimeError(
            "No Provisional CSV links found. Page structure may have changed.\n"
            "Check: https://www.england.nhs.uk/statistics/statistical-work-areas/iucadc-new-from-april-2021/"
        )

    candidates.sort(key=lambda x: x[0], reverse=True)
    period, period_nice, url = candidates[0]
    print(f"\n  Selected: {period_nice} ({period})")
    print(f"  URL: {url}")
    return period, period_nice, url


# ── STEP 2: ALREADY COMMITTED? ───────────────────────────────────────
def already_committed(period, token):
    api_url = (f"https://api.github.com/repos/{GITHUB_REPO}"
               f"/contents/history/kent-111-{period}.json")
    r = requests.get(api_url, headers={
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json"
    })
    if r.status_code == 200:
        print(f"\nStep 2: history/kent-111-{period}.json exists — skipping.")
        return True
    print(f"\nStep 2: No snapshot for {period} — proceeding.")
    return False


# ── STEP 3: FETCH CSV ─────────────────────────────────────────────────
def fetch_csv(csv_url, period_nice):
    print(f"\nStep 3: Fetching {period_nice} CSV...")
    r = requests.get(csv_url, timeout=60, headers=HEADERS)
    print(f"  HTTP {r.status_code} | Size: {len(r.content):,} bytes")
    if r.status_code != 200:
        raise RuntimeError(f"Download failed: HTTP {r.status_code}\nURL: {csv_url}")
    df = pd.read_csv(StringIO(r.text), low_memory=False)
    print(f"  Rows: {len(df):,} | Columns: {list(df.columns)}")
    return df


# ── STEP 4: EXTRACT KENT METRICS ─────────────────────────────────────
def extract_metrics(df, period, period_nice, csv_url):
    # Filter Kent, Medway & Sussex contract
    kent_mask = df['CONTRACT_NAME'].str.contains('Kent', case=False, na=False)
    kent = df[kent_mask]

    if len(kent) == 0:
        # Try ORG_NAME as fallback
        kent_mask = df['ORG_NAME'].str.contains('South East Coast', case=False, na=False)
        kent = df[kent_mask]

    if len(kent) == 0:
        print("  All contract names:", df['CONTRACT_NAME'].unique()[:10])
        raise RuntimeError("Cannot find Kent contract rows.")

    print(f"\n  Contract: {kent['CONTRACT_NAME'].iloc[0]} ({len(kent)} rows)")
    print(f"  Period: {kent['DATE'].min()} to {kent['DATE'].max()}")

    # Sum all values per item code across the full month
    totals = kent.groupby('ITEM_NUMBER')['VALUE'].sum()

    def get(code):
        return int(totals.get(code, 0))

    # Raw SECAmb contract totals
    raw_calls = get('A01')   # calls received
    raw_answd = get('A03')   # calls answered
    raw_amb   = get('E02')   # ambulance dispatched
    raw_ed    = get('E03')   # ED referrals
    raw_gp    = get('G03')   # GP/primary care referrals
    raw_treat = get('E18')   # treated/advised (no escalation)

    print(f"\n  SECAmb contract raw totals (Kent, Medway & Sussex):")
    print(f"    A01 calls received:  {raw_calls:>8,}")
    print(f"    A03 calls answered:  {raw_answd:>8,}")
    print(f"    E02 amb dispatched:  {raw_amb:>8,}")
    print(f"    E03 ED referrals:    {raw_ed:>8,}")
    print(f"    G03 GP referrals:    {raw_gp:>8,}")
    print(f"    E18 treated/advised: {raw_treat:>8,}")

    # Apply Kent ICB population weighting (~40% of contract area)
    def kent_est(n):
        return round(n * KENT_WEIGHT)

    k_calls = kent_est(raw_calls)
    k_answd = kent_est(raw_answd)
    k_amb   = kent_est(raw_amb)
    k_ed    = kent_est(raw_ed)
    k_gp    = kent_est(raw_gp)
    k_treat = kent_est(raw_treat)

    # Rates per 1,000 Kent ICB population per month
    def rate(n):
        return round((n / KENT_ICB_POP) * 1000, 1) if n else 0

    k_call_rate = rate(k_calls)
    k_amb_rate  = rate(k_amb)
    k_ed_rate   = rate(k_ed)

    print(f"\n  Kent ICB estimates (×{KENT_WEIGHT} weighting):")
    print(f"    {'Metric':<22} {'Estimate':>10}  {'Rate/1k':>8}  {'Eng/1k':>8}  {'Ratio':>7}")
    print(f"    {'-'*60}")

    def ratio(k, e):
        return round(k / e, 3) if k and e else None

    rows = [
        ('Calls received', k_calls, k_call_rate, ENG_CALL_RATE_MONTHLY),
        ('Amb dispatched', k_amb,   k_amb_rate,  ENG_AMB_RATE_MONTHLY),
        ('ED referrals',   k_ed,    k_ed_rate,   ENG_ED_RATE_MONTHLY),
    ]
    for name, k, kr, er in rows:
        rv = ratio(kr, er)
        print(f"    {name:<22} {k:>10,}  {kr:>8}  {er:>8}  {str(rv):>7}")

    # QA
    print(f"\n  QA checks:")
    for name, kr, er in [('call', k_call_rate, ENG_CALL_RATE_MONTHLY),
                          ('amb',  k_amb_rate,  ENG_AMB_RATE_MONTHLY),
                          ('ED',   k_ed_rate,   ENG_ED_RATE_MONTHLY)]:
        if er and kr:
            rv = kr / er
            flag = "*** implausible" if rv > 4 or rv < 0.2 else "plausible OK"
            print(f"    {name} rate: {rv:.3f}x England — {flag}")

    return {
        "meta": {
            "generated":        datetime.now(timezone.utc).isoformat(),
            "description":      "NHS 111 demand — Kent & Medway ICB (estimated)",
            "period":           period,
            "period_nice":      period_nice,
            "icb":              "NHS Kent and Medway ICB (QKS)",
            "icb_pop":          KENT_ICB_POP,
            "secamb_contract":  "Kent, Medway & Sussex (111AI9)",
            "kent_weight":      KENT_WEIGHT,
            "source":           "NHS England IUCADC — Provisional Aggregated Raw Data",
            "source_url":       csv_url,
            "note":             (
                f"SECAmb contract covers Kent, Medway & Sussex. "
                f"Kent ICB figures estimated at {int(KENT_WEIGHT*100)}% of contract total "
                f"based on population share. Item codes: A01=calls, E02=amb, E03=ED, G03=GP."
            ),
        },
        "kent_icb_estimated": {
            "calls_received":     k_calls,
            "calls_answered":     k_answd,
            "amb_dispatched":     k_amb,
            "ed_referrals":       k_ed,
            "gp_referrals":       k_gp,
            "treated_advised":    k_treat,
            "call_rate_per_1000": k_call_rate,
            "amb_rate_per_1000":  k_amb_rate,
            "ed_rate_per_1000":   k_ed_rate,
        },
        "secamb_contract_raw": {
            "calls_received":  raw_calls,
            "calls_answered":  raw_answd,
            "amb_dispatched":  raw_amb,
            "ed_referrals":    raw_ed,
            "gp_referrals":    raw_gp,
            "treated_advised": raw_treat,
        },
        "england_reference_monthly": {
            "call_rate_per_1000": ENG_CALL_RATE_MONTHLY,
            "amb_rate_per_1000":  ENG_AMB_RATE_MONTHLY,
            "ed_rate_per_1000":   ENG_ED_RATE_MONTHLY,
            "note": "Annualised 2024/25 IUCADC national rates divided by 12",
        },
        "ratios": {
            "call_rate": ratio(k_call_rate, ENG_CALL_RATE_MONTHLY),
            "amb_rate":  ratio(k_amb_rate,  ENG_AMB_RATE_MONTHLY),
            "ed_rate":   ratio(k_ed_rate,   ENG_ED_RATE_MONTHLY),
        },
    }


# ── STEP 5: COMMIT ────────────────────────────────────────────────────
def commit_file(content, filepath, message, token):
    api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    headers = {"Authorization": f"token {token}",
               "Accept": "application/vnd.github.v3+json"}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None
    payload = {"message": message, "content": b64, "branch": "main"}
    if sha: payload["sha"] = sha
    r = requests.put(api_url, headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"  ✓ {filepath}")
        return True
    print(f"  ✗ {filepath}: {r.status_code} — {r.json().get('message','')}")
    return False


# ── MAIN ──────────────────────────────────────────────────────────────
def main():
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("ASSISTIV_GITHUB_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN").split("\n")[0].strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError("No GitHub token found.")

    period, period_nice, csv_url = discover_latest_csv()
    if already_committed(period, token): return
    df = fetch_csv(csv_url, period_nice)
    output = extract_metrics(df, period, period_nice, csv_url)

    msg = (f"NHS 111 data — {period_nice} — "
           f"Kent est. {output['kent_icb_estimated']['calls_received']:,} calls")
    print(f"\nStep 5: Committing...")
    commit_file(output, "kent-111-data.json", msg, token)
    commit_file(output, f"history/kent-111-{period}.json", msg, token)
    print(f"\nDone — {period_nice}")
    print(f"  Kent est. calls: {output['kent_icb_estimated']['calls_received']:,}")
    print(f"  Call rate/1k:    {output['kent_icb_estimated']['call_rate_per_1000']}")


if __name__ == "__main__":
    main()


---
## ── CBI SECTION ──────────────────────────────────────────────────────
### Carer Burden Index
5 signals · 13 Kent districts · ONS Census 2021 + NHS Fingertips
Commits `kent-cbi-data.json`

**Signal weights:**
- ONS Census 2021 — 20+ hr carers % (district level) — **30%**
- NHS Fingertips 90638 — Carer social isolation (county) — **25%**
- NHS Fingertips 90279 — Carer support info access (county) — **15%**
- NHS Fingertips 93014 — Carer's Assessment rate (county) — **15%**
- NHS Fingertips 90283 — Carer wellbeing score (county) — **15%**

### Cell 4 — CBI: Configuration & Census 2021 Data

In [ ]:
import subprocess
subprocess.run(['pip','install','fingertips_py','requests','pandas','--quiet'], check=True)
print('Dependencies ready')

import requests, pandas as pd, fingertips_py as ftp
import json, base64
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = 'silegrand/assistivagents'
GITHUB_FILE  = 'kent-cbi-data.json'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f'GitHub token: {"Found" if GITHUB_TOKEN else "MISSING"}')

KENT_COUNTY = 'E10000016'  # Kent County Council — finest geography for carer indicators
ENGLAND     = 'E92000001'
BASE_FT     = 'https://fingertips.phe.org.uk/api'

# District name map — ONS code → display name
DISTRICTS = {
    'E07000114': 'Thanet',
    'E07000112': 'Folkestone & Hythe',
    'E07000108': 'Dover',
    'E07000113': 'Swale',
    'E06000035': 'Medway',
    'E07000109': 'Gravesham',
    'E07000105': 'Ashford',
    'E07000106': 'Canterbury',
    'E07000107': 'Dartford',
    'E07000110': 'Maidstone',
    'E07000115': 'Tonbridge & Malling',
    'E07000111': 'Sevenoaks',
    'E07000116': 'Tunbridge Wells',
}
ALL_DISTRICTS = list(DISTRICTS.values())

# ── Signal weights — sum to 1.0 ───────────────────────────────────────────────
WEIGHTS = {
    'census_20plus':    0.30,  # ONS Census 2021 — district level, primary differentiator
    'social_isolation': 0.25,  # Fingertips 90638 — strongest single burnout predictor
    'support_access':   0.15,  # Fingertips 90279 — support infrastructure
    'assessment_rate':  0.15,  # Fingertips 93014 — statutory system engagement
    'wellbeing':        0.15,  # Fingertips 90283 — direct carer wellbeing
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, f'Weights sum to {sum(WEIGHTS.values()):.4f}'
print(f'Weights check: {sum(WEIGHTS.values()):.4f} ✓')

# ── Census 2021 TS039 — pre-extracted district data ───────────────────────────
# Source: ONS Census 2021 TS039 LTLA bulk download (Nomis)
# England average 20+ hr carers: 4.5187% of population aged 5+
ENG_20HRS_PCT = 4.5187

CENSUS_DATA = {
    'Thanet':              {'total':133253,'hrs_9_less':4100, 'hrs_10_19':1772,'hrs_20_49':2899,'hrs_50_plus':5076,'high_hrs':7975, 'pct_20plus':5.9849},
    'Folkestone & Hythe':  {'total':104645,'hrs_9_less':3537, 'hrs_10_19':1446,'hrs_20_49':2168,'hrs_50_plus':3671,'high_hrs':5839, 'pct_20plus':5.5798},
    'Swale':               {'total':142781,'hrs_9_less':4339, 'hrs_10_19':1722,'hrs_20_49':2884,'hrs_50_plus':4984,'high_hrs':7868, 'pct_20plus':5.5105},
    'Dover':               {'total':110741,'hrs_9_less':3791, 'hrs_10_19':1574,'hrs_20_49':2344,'hrs_50_plus':3794,'high_hrs':6138, 'pct_20plus':5.5427},
    'Gravesham':           {'total':100299,'hrs_9_less':2933, 'hrs_10_19':1208,'hrs_20_49':1876,'hrs_50_plus':2969,'high_hrs':4845, 'pct_20plus':4.8306},
    'Medway':              {'total':262465,'hrs_9_less':7265, 'hrs_10_19':3041,'hrs_20_49':4932,'hrs_50_plus':7380,'high_hrs':12312,'pct_20plus':4.6909},
    'Ashford':             {'total':125137,'hrs_9_less':3963, 'hrs_10_19':1487,'hrs_20_49':2199,'hrs_50_plus':3501,'high_hrs':5700, 'pct_20plus':4.5550},
    'Canterbury':          {'total':150606,'hrs_9_less':5003, 'hrs_10_19':1740,'hrs_20_49':2478,'hrs_50_plus':4163,'high_hrs':6641, 'pct_20plus':4.4095},
    'Maidstone':           {'total':165392,'hrs_9_less':5484, 'hrs_10_19':1886,'hrs_20_49':2567,'hrs_50_plus':4245,'high_hrs':6812, 'pct_20plus':4.1187},
    'Tonbridge & Malling': {'total':124520,'hrs_9_less':4341, 'hrs_10_19':1384,'hrs_20_49':1947,'hrs_50_plus':3158,'high_hrs':5105, 'pct_20plus':4.0997},
    'Dartford':            {'total':108471,'hrs_9_less':3107, 'hrs_10_19':1155,'hrs_20_49':1700,'hrs_50_plus':2701,'high_hrs':4401, 'pct_20plus':4.0573},
    'Sevenoaks':           {'total':113792,'hrs_9_less':4234, 'hrs_10_19':1304,'hrs_20_49':1648,'hrs_50_plus':2797,'high_hrs':4445, 'pct_20plus':3.9062},
    'Tunbridge Wells':     {'total':109144,'hrs_9_less':3825, 'hrs_10_19':1127,'hrs_20_49':1410,'hrs_50_plus':2253,'high_hrs':3663, 'pct_20plus':3.3561},
}

print(f'Config loaded | {len(DISTRICTS)} districts | {len(WEIGHTS)} signals')
print(f'Census data: {len(CENSUS_DATA)} districts embedded (ONS Census 2021 TS039)')
print(f'England 20+ hr carer average: {ENG_20HRS_PCT}%')

### Cell 5 — CBI: Fetch Fingertips Carer Indicators

In [ ]:
def ft_fetch(ind_id, area_code):
    """Fetch latest value for indicator at area_code via Fingertips REST API."""
    try:
        r = requests.get(
            f'{BASE_FT}/latest_data/by_indicator_ids'
            f'?indicator_ids={ind_id}&area_codes={area_code}',
            timeout=15
        ).json()
        rows = sorted(r.get(area_code, []), key=lambda x: x.get('TimePeriodSortable', 0))
        if not rows: return None, None
        return rows[-1].get('Value'), rows[-1].get('TimePeriod', '?')
    except Exception as e:
        return None, str(e)

def ft_fetch_fpy(ind_id, area_code):
    """Fallback: fingertips_py with area code filter."""
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None: return None, 'None returned'
        rows = data[data['Area Code'] == area_code].sort_values('Time period')
        if len(rows) == 0: return None, f'no rows for {area_code}'
        return float(rows.tail(1)['Value'].values[0]), str(rows.tail(1)['Time period'].values[0])
    except Exception as e:
        return None, str(e)

FT_INDICATORS = {
    90638: {'key':'social_isolation', 'label':'Carer social isolation',      'invert':True},
    90279: {'key':'support_access',   'label':'Carer support info access',   'invert':True},
    93014: {'key':'assessment_rate',  'label':"Carer's Assessment rate",     'invert':False},
    90283: {'key':'wellbeing',        'label':'Carer wellbeing score',        'invert':False},
}

fingertips = {}
print('Fetching Fingertips carer indicators...')
print(f'{"ID":<8} {"Label":<32} {"Kent":>8}  {"England":>8}  {"Δ%":>8}  Period    Method')
print('-' * 85)

for ind_id, meta in FT_INDICATORS.items():
    # Fetch Kent
    kent_val, kent_period = ft_fetch(ind_id, KENT_COUNTY)
    method = 'REST'
    if kent_val is None:
        kent_val, kent_period = ft_fetch_fpy(ind_id, KENT_COUNTY)
        method = 'ftp_py'

    # Fetch England
    eng_val, _ = ft_fetch(ind_id, ENGLAND)
    if eng_val is None:
        eng_val, _ = ft_fetch_fpy(ind_id, ENGLAND)

    delta = f'{((kent_val-eng_val)/eng_val)*100:+.1f}%' if kent_val and eng_val else 'n/a'

    # For inverted signals: lower kent = worse
    if kent_val and eng_val:
        worse = (kent_val < eng_val) if meta['invert'] else (kent_val > eng_val)
        flag  = '⚠' if worse else '✓'
    else:
        flag = '?'

    print(f'{ind_id:<8} {meta["label"]:<32} '
          f'{str(round(kent_val,2)) if kent_val else "n/a":>8}  '
          f'{str(round(eng_val,2)) if eng_val else "n/a":>8}  '
          f'{delta:>8}  {str(kent_period):<10}  {flag} [{method}]')

    fingertips[meta['key']] = {
        'indicator_id': ind_id,
        'label':        meta['label'],
        'kent_val':     kent_val,
        'england_val':  eng_val,
        'period':       kent_period,
        'invert':       meta['invert'],
        'source':       f'NHS Fingertips indicator {ind_id} — Kent County (E10000016)',
        'geo_note':     'County level — applied uniformly across all 13 districts',
    }

fetched = sum(1 for v in fingertips.values() if v['kent_val'])
print(f'\n{fetched}/{len(FT_INDICATORS)} Fingertips indicators fetched')
print('Note: all at Kent County (E10000016) — finest available geography for these indicators')

### Cell 6 — CBI: Build District Scores · Commit

In [ ]:
def norm(kent_val, eng_val, invert=False):
    """Normalise to 0-100 where 50 = England average.
    Invert=True: lower value = worse = higher score (used for social isolation etc.)"""
    if not kent_val or not eng_val: return 50.0
    ratio = kent_val / eng_val
    if invert:
        # Lower than England = higher burden — invert the ratio
        score = (2 - ratio) * 50
    else:
        score = ratio * 50
    return round(min(100, max(0, score)), 1)

def norm_census(district_pct, eng_pct):
    """Normalise 20+ hr carer % against England average."""
    if not district_pct or not eng_pct: return 50.0
    return round(min(100, max(0, (district_pct / eng_pct) * 50)), 1)

# ── County-level Fingertips normalised scores ─────────────────────────────────
# Same for all districts — county signals shift the baseline uniformly
ft_scores = {}
for key, v in fingertips.items():
    ft_scores[key] = norm(v['kent_val'], v['england_val'], invert=v['invert'])

print('County-level Fingertips normalised scores (applied to all districts):')
for key, score in ft_scores.items():
    label = fingertips[key]['label']
    w     = WEIGHTS[key]
    contribution = round(score * w, 2)
    print(f'  {label:<35} {score:>6.1f}  weight:{w:.0%}  contribution:{contribution:.1f}')
ft_county_contribution = sum(ft_scores.get(k, 50) * WEIGHTS[k]
                             for k in ['social_isolation','support_access','assessment_rate','wellbeing'])
print(f'\nTotal county-level contribution to CBI (before Census): {ft_county_contribution:.1f}')
print(f'Census (district-level) contributes remaining 30% — drives within-county spread')

# ── LAD codes ─────────────────────────────────────────────────────────────────
LAD_CODES = {
    'Thanet':'E07000114','Folkestone & Hythe':'E07000112','Dover':'E07000108',
    'Swale':'E07000113','Medway':'E06000035','Gravesham':'E07000109',
    'Ashford':'E07000105','Canterbury':'E07000106','Dartford':'E07000107',
    'Maidstone':'E07000110','Tonbridge & Malling':'E07000115',
    'Sevenoaks':'E07000111','Tunbridge Wells':'E07000116',
}

# ── Build district scores ──────────────────────────────────────────────────────
districts = []
print(f'\n{"District":<25} {"Census":>7}  {"FT":>7}  {"CBI":>6}  {"Tier":<10}  20+%  50+carers')
print('-' * 80)

for name in ALL_DISTRICTS:
    c = CENSUS_DATA[name]
    census_score = norm_census(c['pct_20plus'], ENG_20HRS_PCT)

    # Weighted CBI
    signal_scores = {
        'census_20plus':    census_score,
        'social_isolation': ft_scores.get('social_isolation', 50),
        'support_access':   ft_scores.get('support_access', 50),
        'assessment_rate':  ft_scores.get('assessment_rate', 50),
        'wellbeing':        ft_scores.get('wellbeing', 50),
    }
    cbi = round(sum(signal_scores[k] * WEIGHTS[k] for k in WEIGHTS))
    tier = ('critical' if cbi >= 70 else 'high' if cbi >= 55
            else 'moderate' if cbi >= 40 else 'low')

    ft_total = round(sum(ft_scores.get(k,50)*WEIGHTS[k]
                         for k in ['social_isolation','support_access','assessment_rate','wellbeing']), 1)

    print(f'{name:<25} {census_score:>7.1f}  {ft_total:>7.1f}  {cbi:>6}  {tier:<10}  '
          f'{c["pct_20plus"]:>4.2f}%  {c["hrs_50_plus"]:,}')

    districts.append({
        'name':           name,
        'lad_code':       LAD_CODES[name],
        'cbi':            cbi,
        'tier':           tier,
        'signal_scores':  signal_scores,
        'signal_weights': WEIGHTS,
        'census': {
            'pct_20plus':       c['pct_20plus'],
            'high_hrs_carers':  c['high_hrs'],
            'hrs_50_plus':      c['hrs_50_plus'],
            'total_pop_5plus':  c['total'],
            'england_avg_pct':  ENG_20HRS_PCT,
            'source':           'ONS Census 2021 TS039 — LTLA level',
        },
    })

districts.sort(key=lambda x: x['cbi'], reverse=True)

# ── Print quadrant summary ─────────────────────────────────────────────────────
print('\n── Quadrant summary (requires FEP scores from kent-fep-data.json) ──')
print('Load kent-fep-data.json to compute FEP × CBI quadrants.')
print('High CBI districts for reference:')
for d in districts:
    if d['tier'] in ('high','critical'):
        print(f'  {d["name"]:<25} CBI:{d["cbi"]}  {d["tier"]}')

# ── Assemble output JSON ────────────────────────────────────────────────────────
output = {
    'meta': {
        'generated':      datetime.now(timezone.utc).isoformat(),
        'description':    'Kent & Medway Carer Burden Index — Assistiv Systems Layer 2 companion',
        'version':        '1.0',
        'icb':            'NHS Kent and Medway ICB (QKS)',
        'icb_ons_code':   'E54000032',
        'signals': {
            'census_20plus':    {'weight':0.30,'geography':'district','source':'ONS Census 2021 TS039'},
            'social_isolation': {'weight':0.25,'geography':'county',  'source':'NHS Fingertips 90638'},
            'support_access':   {'weight':0.15,'geography':'county',  'source':'NHS Fingertips 90279'},
            'assessment_rate':  {'weight':0.15,'geography':'county',  'source':'NHS Fingertips 93014'},
            'wellbeing':        {'weight':0.15,'geography':'county',  'source':'NHS Fingertips 90283'},
        },
        'methodology': (
            'Signals normalised to 0-100 (50=England average, >50=higher burden). '
            'County-level Fingertips signals applied uniformly across all 13 districts. '
            'District variation driven by ONS Census 2021 20+ hr carer rate (30% weight). '
            'Antidepressant/anxiolytic prescribing excluded — already in FEP model. '
            'DWP Carer Allowance pending — add as Signal 6 when available.'
        ),
        'limitations': (
            'Fingertips carer indicators not available below county level for Kent. '
            'CBI is a population-level triage index, not an individual diagnostic tool. '
            'Census data from 2021 — carer burden has likely increased since then.'
        ),
        'note_on_fep_cbi': (
            'CBI is designed to be read alongside the FEP score, not instead of it. '
            'High FEP + High CBI = frailty risk AND carer system under strain simultaneously. '
            'Use the quadrant map for commissioning decisions.'
        ),
    },
    'county_baseline': fingertips,
    'england_census_avg_20plus_pct': ENG_20HRS_PCT,
    'districts': districts,
}

# ── Commit to GitHub ──────────────────────────────────────────────────────────
def commit_to_github(content, repo, filepath, token):
    api = f'https://api.github.com/repos/{repo}/contents/{filepath}'
    hdrs = {'Authorization': f'token {token}',
            'Accept': 'application/vnd.github.v3+json'}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api, headers=hdrs)
    sha = r.json().get('sha') if r.status_code == 200 else None
    payload = {
        'message': f'CBI v1.0 — {datetime.now(timezone.utc).strftime("%Y-%m-%d")} — 5 signals',
        'content': b64, 'branch': 'main'
    }
    if sha: payload['sha'] = sha
    r = requests.put(api, headers=hdrs, json=payload)
    if r.status_code in (200, 201):
        print(f'Committed ✓')
        print(f'  https://raw.githubusercontent.com/{repo}/main/{filepath}')
        print(f'  {r.json().get("commit",{}).get("html_url","")}')
        return True
    print(f'Commit failed: {r.status_code} — {r.json().get("message","")}')
    return False

print('\nCommitting kent-cbi-data.json to GitHub...')
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)
print('\nDone. Refresh quarterly when new Census or Fingertips data is published.')
print(f'Districts scored: {len(districts)}')
high = sum(1 for d in districts if d["tier"] in ("high","critical"))
print(f'High/critical CBI districts: {high}/13')

---
## ── RAVI SECTION ─────────────────────────────────────────────────────
### Rural Access Vulnerability Index
5 signals · 1,065 Kent LSOAs · IMD 2019 + ONS RUC + Census 2021
Commits `kent-ravi-data.json`

**Run order within this section: Cell 7 → 8 → 9 → 10**
Each cell loads the current JSON from GitHub and modifies only its own fields.
Safe to re-run individual cells independently.

**Signal weights:**
- IMD 2019 Geographic Barriers sub-domain (road distance to GP etc) — **30%**
- ONS 2021 Rural Urban Classification — Urban → Hamlet/Isolated — **25%**
- Population aged 65+ (Census 2021 Nomis) — **20%**
- No car or van in household (Census 2021 Nomis) — **15%**
- Day-to-day activities limited a lot (Census 2021 Nomis) — **10%**

### Cell 7 — RAVI: Install Additional Dependencies

In [ ]:
!pip install requests pandas openpyxl --quiet
print("Dependencies installed ✓")

### Cell 8 — RAVI: Data Pipeline · Scores · Commit

In [ ]:
"""
RAVI Cell 2 — Data pipeline: IMD 2019, RUC 2021, Census 2021 → RAVI scores → GitHub
"""
import requests, pandas as pd, json, base64, io
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()

KENT_LAD_CODES = [
    'E07000105','E07000106','E07000107','E07000108','E07000109',
    'E07000110','E06000035','E07000111','E07000112','E07000113',
    'E07000114','E07000115','E07000116',
]
WEIGHTS = {'geo_barriers':0.30,'ruc_score':0.25,'pct_65plus':0.20,'pct_no_car':0.15,'pct_llti':0.10}
assert abs(sum(WEIGHTS.values())-1.0)<0.001
LAD_NAMES = {
    'E07000105':'Ashford','E07000106':'Canterbury','E07000107':'Dartford',
    'E07000108':'Dover','E07000109':'Gravesham','E07000110':'Maidstone',
    'E06000035':'Medway','E07000111':'Sevenoaks','E07000112':'Folkestone & Hythe',
    'E07000113':'Swale','E07000114':'Thanet','E07000115':'Tonbridge & Malling',
    'E07000116':'Tunbridge Wells',
}
ENG_FALLBACKS = {'pct_65plus':18.4,'pct_no_car':25.6,'pct_llti':17.8}
HEADERS = {"User-Agent":"Mozilla/5.0 AssistivSystems/1.0"}
print(f"Weights: {sum(WEIGHTS.values()):.2f} ✓")

# ── IMD 2019 ──────────────────────────────────────────────────────────
print("\n── Step 1: IMD 2019 Geographic Barriers ──")
IMD_URL = ("https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/"
           "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv")
r = requests.get(IMD_URL, timeout=120, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
if r.status_code != 200:
    raise RuntimeError("IMD download failed. See: https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019")
imd_raw = pd.read_csv(io.StringIO(r.text))
geo_col = next((c for c in imd_raw.columns if 'Geographical Barriers' in c and 'Score' in c), None)
if not geo_col: raise ValueError(f"Geo barriers column not found. Columns: {list(imd_raw.columns)}")
print(f"  Geo barriers column: {geo_col!r}")
imd_kent = imd_raw[imd_raw['Local Authority District code (2019)'].isin(KENT_LAD_CODES)][
    ['LSOA code (2011)','LSOA name (2011)','Local Authority District code (2019)',geo_col]
].copy()
imd_kent.columns = ['lsoa_code','lsoa_name','lad_code','geo_barriers_score']
imd_kent = imd_kent.dropna(subset=['geo_barriers_score']).reset_index(drop=True)
print(f"  Kent LSOAs: {len(imd_kent)} | Range: {imd_kent.geo_barriers_score.min():.3f} – {imd_kent.geo_barriers_score.max():.3f}")

# ── RUC 2021 ──────────────────────────────────────────────────────────
print("\n── Step 2: Rural Urban Classification ──")
r = requests.get("https://raw.githubusercontent.com/mysociety/uk_ruc/main/output/composite_ruc.csv", timeout=30, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
if r.status_code == 200:
    ruc_raw = pd.read_csv(io.StringIO(r.text))
    ruc_col = 'ukruc-3' if 'ukruc-3' in ruc_raw.columns else 'ukruc-2'
    code_map  = {0:'Urban',1:'Rural town/village',2:'Hamlet/Isolated'} if ruc_col=='ukruc-3' else {0:'Urban',1:'Rural'}
    score_map = {0:0,1:55,2:90} if ruc_col=='ukruc-3' else {0:10,1:70}
    ruc_kent = ruc_raw[ruc_raw['lsoa'].isin(imd_kent.lsoa_code)][['lsoa',ruc_col]].copy()
    ruc_kent.columns = ['lsoa_code','ruc_raw']
    ruc_kent['ruc_label'] = ruc_kent.ruc_raw.map(code_map).fillna('Unknown')
    ruc_kent['ruc_score'] = ruc_kent.ruc_raw.map(score_map).fillna(30.0)
    imd_kent = imd_kent.merge(ruc_kent[['lsoa_code','ruc_label','ruc_score']], on='lsoa_code', how='left')
    imd_kent['ruc_label'] = imd_kent['ruc_label'].fillna('Unknown')
    imd_kent['ruc_score'] = imd_kent['ruc_score'].fillna(30.0)
    print(f"  Matched: {imd_kent.ruc_label.ne('Unknown').sum()}/{len(imd_kent)}")
    print(f"  Distribution:\n{imd_kent.ruc_label.value_counts().to_string()}")
else:
    print("  WARNING: RUC unavailable — using 30 for all")
    imd_kent['ruc_label'] = 'Unknown'; imd_kent['ruc_score'] = 30.0

# ── CENSUS 2021 ───────────────────────────────────────────────────────
print("\n── Step 3: Census 2021 via Nomis ──")
def fetch_nomis(dataset_id, params, description, col, fallback):
    url = (f"https://www.nomisweb.co.uk/api/v01/dataset/{dataset_id}.data.csv"
           f"?date=latest&geography=TYPE297&measures=20301&{params}&select=GEOGRAPHY_CODE,OBS_VALUE")
    print(f"  {description}...")
    try:
        r = requests.get(url, timeout=60, headers=HEADERS)
        print(f"    HTTP {r.status_code} | {len(r.content):,} bytes")
        if r.status_code==200 and len(r.content)>500:
            df = pd.read_csv(io.StringIO(r.text))
            df.columns = ['lsoa_code', col]
            df = df[df.lsoa_code.isin(imd_kent.lsoa_code)]
            if len(df)>100:
                df = df.groupby('lsoa_code')[col].sum().reset_index()
                print(f"    ✓ {len(df)} LSOAs | Mean: {df[col].mean():.1f}%")
                return df
        print(f"    Using England average {fallback}%")
    except Exception as e:
        print(f"    Error: {e} — using {fallback}%")
    return None

age_df  = fetch_nomis('NM_2010_1','c2021_age_h=7,8,9,10,11,12,13,14,15,16,17,18','Age 65+','pct_65plus',ENG_FALLBACKS['pct_65plus'])
car_df  = fetch_nomis('NM_2072_1','c2021_carsvan_4=1','No car','pct_no_car',ENG_FALLBACKS['pct_no_car'])
llti_df = fetch_nomis('NM_2064_1','c2021_disability_3=1','LLTI','pct_llti',ENG_FALLBACKS['pct_llti'])
for df, col in [(age_df,'pct_65plus'),(car_df,'pct_no_car'),(llti_df,'pct_llti')]:
    if df is not None:
        imd_kent = imd_kent.merge(df, on='lsoa_code', how='left')
    else:
        imd_kent[col] = None
    imd_kent[col] = imd_kent[col].fillna(ENG_FALLBACKS[col])

# ── RAVI SCORE ────────────────────────────────────────────────────────
print("\n── Step 4: RAVI scores ──")
def norm(s):
    mn,mx = s.min(),s.max()
    return pd.Series([50.0]*len(s),index=s.index) if mx==mn else ((s-mn)/(mx-mn)*100).round(1)
imd_kent['geo_barriers_norm'] = norm(imd_kent.geo_barriers_score)
imd_kent['ruc_norm']          = norm(imd_kent.ruc_score)
imd_kent['age_norm']          = norm(imd_kent.pct_65plus)
imd_kent['car_norm']          = norm(imd_kent.pct_no_car)
imd_kent['llti_norm']         = norm(imd_kent.pct_llti)
imd_kent['ravi'] = (
    imd_kent.geo_barriers_norm*WEIGHTS['geo_barriers'] + imd_kent.ruc_norm*WEIGHTS['ruc_score'] +
    imd_kent.age_norm*WEIGHTS['pct_65plus'] + imd_kent.car_norm*WEIGHTS['pct_no_car'] +
    imd_kent.llti_norm*WEIGHTS['pct_llti']
).round(1)
def band(s): return 'critical' if s>=70 else 'high' if s>=55 else 'moderate' if s>=40 else 'low'
imd_kent['ravi_band'] = imd_kent.ravi.apply(band)
imd_kent['district']  = imd_kent.lad_code.map(LAD_NAMES)
print(f"RAVI: {imd_kent.ravi.min():.1f} – {imd_kent.ravi.max():.1f} | Mean: {imd_kent.ravi.mean():.1f}")
print(f"Risk bands:\n{imd_kent.ravi_band.value_counts().to_string()}")

# ── DISTRICT SUMMARY ──────────────────────────────────────────────────
district_summary = {}
for dist, grp in imd_kent.groupby('district'):
    high = grp[grp.ravi>=55]
    district_summary[dist] = {
        'lsoa_count':int(len(grp)), 'mean_ravi':round(float(grp.ravi.mean()),1),
        'max_ravi':round(float(grp.ravi.max()),1), 'high_ravi_count':int(len(high)),
        'pct_high':round(len(high)/len(grp)*100,1),
        'top_lsoa':str(grp.nlargest(1,'ravi').iloc[0]['lsoa_name']),
    }
print(f"\n{'District':<25} {'Mean':>5}  {'Max':>5}  {'High/Crit':>9}")
for d,s in sorted(district_summary.items(),key=lambda x:-x[1]['mean_ravi']):
    print(f"  {d:<23} {s['mean_ravi']:>5.1f}  {s['max_ravi']:>5.1f}  {s['high_ravi_count']:>9}")

# ── BUILD AND COMMIT ──────────────────────────────────────────────────
# Grid placeholder coords — Cell 3 replaces with precise ONS centroids
DC = {
    'Ashford':(51.148,0.874),'Canterbury':(51.279,1.076),'Dartford':(51.446,0.221),
    'Dover':(51.126,1.313),'Folkestone & Hythe':(51.083,1.167),'Gravesham':(51.442,0.368),
    'Maidstone':(51.272,0.523),'Medway':(51.385,0.523),'Sevenoaks':(51.272,0.187),
    'Swale':(51.333,0.753),'Thanet':(51.358,1.383),'Tonbridge & Malling':(51.197,0.273),
    'Tunbridge Wells':(51.132,0.263),
}
dc = {}
for _, row in imd_kent.iterrows():
    d = row.district
    i = dc.get(d,0); dc[d]=i+1
    c = DC.get(d,(51.27,0.52))
    imd_kent.at[_,'lat'] = round(c[0]-0.07+(i//10)*0.016,5)
    imd_kent.at[_,'lon'] = round(c[1]-0.07+(i%10)*0.016,5)

lsoa_list = [{
    'lsoa_code':row.lsoa_code,'lsoa_name':str(row.lsoa_name),
    'district':str(row.get('district','')),'lad_code':row.lad_code,
    'lat':round(float(row.get('lat',51.27)),5),'lon':round(float(row.get('lon',0.52)),5),
    'ravi':round(float(row.ravi),1),'ravi_band':row.ravi_band,'place_name':'',
    'signals':{
        'geo_barriers_score':round(float(row.geo_barriers_score),3),
        'geo_barriers_norm':round(float(row.geo_barriers_norm),1),
        'ruc_label':str(row.get('ruc_label','Unknown')),
        'ruc_score':round(float(row.get('ruc_score',30)),1),
        'ruc_norm':round(float(row.ruc_norm),1),
        'pct_65plus':round(float(row.pct_65plus),1),'age_norm':round(float(row.age_norm),1),
        'pct_no_car':round(float(row.pct_no_car),1),'car_norm':round(float(row.car_norm),1),
        'pct_llti':round(float(row.pct_llti),1),'llti_norm':round(float(row.llti_norm),1),
    }
} for _,row in imd_kent.iterrows()]

output = {
    'meta':{
        'generated':datetime.now(timezone.utc).isoformat(),
        'description':'Rural Access Vulnerability Index — Kent & Medway LSOAs',
        'version':'1.4','lsoa_count':len(lsoa_list),
        'sources':{'geo_barriers':'IMD 2019 File 7 (MHCLG)','ruc':'mySociety/ONS RUC 2021',
                   'census':'Census 2021 Nomis or England average fallback'},
        'weights':WEIGHTS,'census_source':'real' if age_df is not None else 'england_average_fallback',
        'coord_method':'grid placeholder — run Cell 3','has_coordinates':True,'has_place_names':False,
    },
    'district_summary':district_summary,'lsoas':lsoa_list,
}

def commit(content, filepath, token, msg):
    api=f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs={"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64=base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r=requests.get(api,headers=hdrs); sha=r.json().get("sha") if r.status_code==200 else None
    pay={"message":msg,"content":b64,"branch":"main"}
    if sha: pay["sha"]=sha
    r=requests.put(api,headers=hdrs,json=pay)
    print(f"  {'✓' if r.status_code in(200,201) else '✗'} {filepath}")
    return r.status_code in(200,201)

commit(output, GITHUB_FILE, GITHUB_TOKEN,
       f"RAVI v1.4 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — {len(lsoa_list)} LSOAs")
print(f"\nCell 2 done — run Cell 3 for precise coordinates")


### Cell 9 — RAVI: Add Precise ONS Coordinates

In [ ]:
"""
RAVI Cell 3 — Add precise ONS population-weighted centroids.
Loads current JSON from GitHub, adds lat/lon, recommits.
Does NOT overwrite place_name or any other fields.
"""
import requests, json, base64
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
HEADERS      = {"User-Agent":"Mozilla/5.0 AssistivSystems/1.0"}

print("Loading current JSON from GitHub...")
r    = requests.get(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}", timeout=15)
ravi = r.json()
kent_codes = set(l['lsoa_code'] for l in ravi['lsoas'])
print(f"  {len(ravi['lsoas'])} LSOAs | version: {ravi['meta'].get('version')}")

print("\nFetching ONS centroids (geometry extraction)...")
BASE = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_PopCentroids_EW_2021_V4/FeatureServer/0/query")

centroids = {}
offset = 0
while True:
    r = requests.get(BASE, timeout=30, headers=HEADERS, params={
        'where':'1=1','outFields':'LSOA21CD','returnGeometry':'true',
        'outSR':'4326','f':'json','resultOffset':offset,'resultRecordCount':1000,
    })
    data = r.json()
    if 'error' in data: print(f"  Error: {data['error']}"); break
    features = data.get('features',[])
    if not features: break
    for f in features:
        code = f.get('attributes',{}).get('LSOA21CD')
        geom = f.get('geometry',{})
        if code and geom and geom.get('x') and geom.get('y'):
            centroids[code] = {'lat':round(float(geom['y']),5),'lon':round(float(geom['x']),5)}
    offset += len(features)
    if offset%5000==0: print(f"  {offset:,} fetched...")
    if len(features)<1000: break

matched = {c:centroids[c] for c in kent_codes if c in centroids}
print(f"  Matched: {len(matched)}/{len(kent_codes)} Kent LSOAs")

updated = 0
for lsoa in ravi['lsoas']:
    if lsoa['lsoa_code'] in matched:
        lsoa['lat'] = matched[lsoa['lsoa_code']]['lat']
        lsoa['lon'] = matched[lsoa['lsoa_code']]['lon']
        updated += 1

ravi['meta']['coord_method'] = 'ONS population-weighted centroids — geometry extraction'
ravi['meta']['generated']    = datetime.now(timezone.utc).isoformat()
print(f"  Updated: {updated} LSOAs with precise coordinates")

thanet = [l for l in ravi['lsoas'] if l['district']=='Thanet'][:2]
print(f"\nThanet check (expect ~51.3-51.4, 1.26-1.42):")
for l in thanet: print(f"  {l['lsoa_name']}: {l['lat']}, {l['lon']}")

def commit(content, filepath, token, msg):
    api=f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs={"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64=base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r=requests.get(api,headers=hdrs); sha=r.json().get("sha") if r.status_code==200 else None
    pay={"message":msg,"content":b64,"branch":"main"}
    if sha: pay["sha"]=sha
    r=requests.put(api,headers=hdrs,json=pay)
    print(f"  {'✓' if r.status_code in(200,201) else '✗'} {filepath}")

today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print("\nCommitting...")
commit(ravi, GITHUB_FILE, GITHUB_TOKEN, f"RAVI — {today} — precise centroids ({updated} LSOAs)")
print(f"Cell 3 done — run Cell 4 for place names")


### Cell 10 — RAVI: Add Settlement Place Names

In [ ]:
"""
RAVI Cell 4 — Add ONS settlement place names.
Loads current JSON from GitHub, adds place_name field, recommits.
Does NOT overwrite lat/lon or any other fields.
"""
import requests, json, base64, io
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
HEADERS      = {"User-Agent":"Mozilla/5.0 AssistivSystems/1.0"}

print("Loading current JSON from GitHub...")
r    = requests.get(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}", timeout=15)
ravi = r.json()
kent_codes = set(l['lsoa_code'] for l in ravi['lsoas'])
print(f"  {len(ravi['lsoas'])} LSOAs | coords: {ravi['meta'].get('coord_method','?')[:40]}")

# Verify coordinates are precise before proceeding
sample_lat = ravi['lsoas'][0].get('lat', 0)
if sample_lat in (51.27, 0.0):
    print("\nWARNING: Coordinates look like grid placeholders.")
    print("Run Cell 3 first to add precise coordinates, then re-run this cell.")

print("\nFetching ONS LSOA to Built-Up Area lookup...")
# Confirmed service URL from data.gov.uk metadata
BASE = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA11_BUASD11_BUA11_LAD11_RGN11_EW_LU_133d9317ea7245d288ba6e5d22f0a131/"
        "FeatureServer/0/query")

all_records = {}
offset = 0
batch  = 2000
while True:
    r = requests.get(BASE, timeout=30, headers=HEADERS, params={
        'where':'1=1','outFields':'LSOA11CD,BUASD11NM,BUA11NM',
        'returnGeometry':'false','f':'json',
        'resultOffset':offset,'resultRecordCount':batch,
    })
    print(f"  HTTP {r.status_code} | {len(r.content):,} bytes | offset {offset}")
    if r.status_code!=200: break
    data = r.json()
    if 'error' in data: print(f"  Error: {data['error']}"); break
    features = data.get('features',[])
    if not features: break
    for f in features:
        a = f.get('attributes',{})
        code = a.get('LSOA11CD')
        if code and code in kent_codes:
            all_records[code] = {
                'buasd': str(a.get('BUASD11NM','') or '').strip(),
                'bua':   str(a.get('BUA11NM','')   or '').strip(),
            }
    offset += len(features)
    if len(features)<batch: break

print(f"  Kent LSOAs matched: {len(all_records)}/{len(kent_codes)}")

if len(all_records)==0:
    print("\nLookup unavailable. Download manually from:")
    print("https://geoportal.statistics.gov.uk/datasets/f0aac7ccbfd04cda9eb03e353c613faa/about")
    print("Search: 'LSOA 2011 to Built-up Area Best Fit Lookup' → Download → CSV")
    print("Upload to Colab then run:")
    print("  df = pd.read_csv('your_file.csv')")
    print("  for _, row in df[df.LSOA11CD.isin(kent_codes)].iterrows():")
    print("      all_records[row.LSOA11CD] = {'buasd':str(row.BUASD11NM or ''),'bua':str(row.BUA11NM or '')}")
    raise SystemExit("BUA lookup failed — see instructions above")

def place_name(rec):
    for key in ('buasd','bua'):
        v = rec.get(key,'').strip()
        if v and v not in ('nan','None',''): return v
    return 'Rural (no settlement)'

bua_lookup = {code: place_name(rec) for code, rec in all_records.items()}
n_named = sum(1 for v in bua_lookup.values() if v!='Rural (no settlement)')
n_rural = sum(1 for v in bua_lookup.values() if v=='Rural (no settlement)')
print(f"  Named settlements: {n_named} | Rural: {n_rural}")

# Update ONLY place_name — never touch lat/lon or other fields
for lsoa in ravi['lsoas']:
    lsoa['place_name'] = bua_lookup.get(lsoa['lsoa_code'], 'Unknown')

ravi['meta']['has_place_names'] = True
ravi['meta']['generated']       = datetime.now(timezone.utc).isoformat()

print(f"\n── Top 15 critical/high RAVI LSOAs ──")
print(f"{'LSOA':<25} {'Place':<30} {'District':<22} {'RAVI':>5}")
print("-"*85)
for l in sorted([l for l in ravi['lsoas'] if l['ravi_band'] in ('critical','high')],key=lambda x:-x['ravi'])[:15]:
    print(f"  {l['lsoa_name']:<23} {l['place_name']:<30} {l['district']:<22} {l['ravi']:>5.1f}")

def commit(content, filepath, token, msg):
    api=f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs={"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64=base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r=requests.get(api,headers=hdrs); sha=r.json().get("sha") if r.status_code==200 else None
    pay={"message":msg,"content":b64,"branch":"main"}
    if sha: pay["sha"]=sha
    r=requests.put(api,headers=hdrs,json=pay)
    print(f"  {'✓' if r.status_code in(200,201) else '✗'} {filepath}")

today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print("\nCommitting...")
commit(ravi, GITHUB_FILE, GITHUB_TOKEN,
       f"RAVI — {today} — place names ({n_named} settlements, {n_rural} rural)")
print(f"\nDone — all four cells complete")
print(f"View: https://silegrand.github.io/assistivagents/rural-access-vulnerability.html")
